_Supervised learning for classification_

This leakage-safe workflow splits the raw dataset into training and test sets before any supervised feature selection is performed. All feature-selection steps are fit on the training set only and then applied to the held-out test set. The transformed training and test matrices are then used by the individual model notebooks.

In [ ]:
# Import pachages
from functools import partial
from sklearn.model_selection import train_test_split, StratifiedKFold
import os
import pickle
import pandas as pd
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, mutual_info_classif
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from IPython.display import display

with open("outputs/01_Variables.pkl", 'rb') as file:
    data, labels = pickle.load(file)

data = data.transpose()
y = labels["Group"]

### 2.1 Split raw data into training/test sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data, 
                                                    y, 
                                                    test_size=0.4, 
                                                    random_state=41, 
                                                    stratify=y)
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

### 2.2 Fit feature selection on the training set only

The same feature-selection recipe as the original workflow is retained, but every supervised step is learned using `X_train` only and then applied to `X_test`.

In [ ]:
# 1) Low-variance filtering
percentage_cf = 0.8
variance_selector = VarianceThreshold(threshold=(percentage_cf * (1 - percentage_cf)))
X_train_vt = pd.DataFrame(
    variance_selector.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns[variance_selector.get_support(indices=True)]
)
X_test_vt = X_test.loc[:, X_train_vt.columns]

# 2) Univariate ANOVA selection
k_univariate = max(1, int(X_train_vt.shape[1] * 0.1))
univariate_selector = SelectKBest(score_func=f_classif, k=k_univariate)
X_train_uni = pd.DataFrame(
    univariate_selector.fit_transform(X_train_vt, y_train),
    index=X_train_vt.index,
    columns=X_train_vt.columns[univariate_selector.get_support(indices=True)]
)
X_test_uni = X_test_vt.loc[:, X_train_uni.columns]

# 3) Mutual-information selection
k_mi = max(1, int(X_train_uni.shape[1] * 0.5))
mi_selector = SelectKBest(score_func=partial(mutual_info_classif, random_state=42), k=k_mi)
X_train_mi = pd.DataFrame(
    mi_selector.fit_transform(X_train_uni, y_train),
    index=X_train_uni.index,
    columns=X_train_uni.columns[mi_selector.get_support(indices=True)]
)
X_test_mi = X_test_uni.loc[:, X_train_mi.columns]

# 4) RFE on scaled training data only
k_rfe = min(50, X_train_mi.shape[1])
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_mi)
X_test_scaled = scaler.transform(X_test_mi)
rfe_model = RandomForestClassifier(n_estimators=100, random_state=42)
rfe_selector = RFE(estimator=rfe_model, n_features_to_select=k_rfe, step=min(50, X_train_mi.shape[1]))
X_train_selected = pd.DataFrame(
    rfe_selector.fit_transform(X_train_scaled, y_train),
    index=X_train_mi.index,
    columns=X_train_mi.columns[rfe_selector.get_support(indices=True)]
)
X_test_selected = pd.DataFrame(
    rfe_selector.transform(X_test_scaled),
    index=X_test_mi.index,
    columns=X_train_selected.columns
)

X_train = X_train_selected
X_test = X_test_selected

print("Train shape after feature selection:", X_train.shape)
print("Test shape after feature selection:\t", X_test.shape)

### 2.3 Stratified cross-validation and save outputs

In [ ]:
kfold = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

directory = 'outputs'
if not os.path.exists(directory):
    os.makedirs(directory)

feature_selection_artifacts = {
    'variance_selector': variance_selector,
    'univariate_selector': univariate_selector,
    'mi_selector': mi_selector,
    'scaler': scaler,
    'rfe_selector': rfe_selector,
    'selected_columns': X_train.columns.to_list()
}
with open('outputs/03_FeatureSelection.pkl', 'wb') as file:
    pickle.dump(feature_selection_artifacts, file)

# Save variables to a file
with open('outputs/03_Variables.pkl', 'wb') as file:
    pickle.dump((X_train, X_test, y_train, y_test, kfold), file)